# 01 — Data Loading, Profiling & Cleaning
**ReadmitScope US** · CMS Hospital Readmissions Reduction Program (FY 2026)

This notebook loads the raw CMS extract, profiles it, documents every data-quality
issue, and writes the cleaned table. Decisions here are mirrored in
`docs/03_data_quality_log.md` and implemented in `src/build_aggregates.py`.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
ORANGE, DARK, MUTED = "#FF8000", "#0D0D0D", "#9CA3AF"
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold",
                     "axes.titlesize": 12, "font.size": 10})

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "hrrp_raw.csv"
PROC = ROOT / "data" / "processed" / "hrrp_reported.csv"

## 1. Load raw data
Numeric fields are read as strings because CMS embeds text suppression markers in them.

In [2]:
raw = pd.read_csv(RAW, dtype=str)
print("shape:", raw.shape)
raw.head(3)

shape: (18330, 12)


,Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-CABG-HRRP,137,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,010001,AL,READM-30-AMI-HRRP,273,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024


## 2. Structure & grain
Confirm one row per hospital × condition, and that `Facility ID` is the real key.

In [3]:
print("rows per facility (expect all == 6):")
print(raw.groupby('Facility ID').size().value_counts())
print("\nunique Facility IDs:", raw['Facility ID'].nunique())
print("unique Facility Names:", raw['Facility Name'].nunique(), "(fewer names than IDs -> some shared names)")
print("states:", raw['State'].nunique())
print("measures:", raw['Measure Name'].unique())

rows per facility (expect all == 6):
6    3055
Name: count, dtype: int64

unique Facility IDs: 3055
unique Facility Names: 2995 (fewer names than IDs -> some shared names)
states: 51
measures: <StringArray>
['READM-30-HIP-KNEE-HRRP',     'READM-30-CABG-HRRP',      'READM-30-AMI-HRRP',
     'READM-30-COPD-HRRP',       'READM-30-PN-HRRP',       'READM-30-HF-HRRP']
Length: 6, dtype: str


## 3. Missingness & suppression audit
CMS suppresses unreliable cells. We quantify it and check whether footnotes *explain* it.

In [4]:
err_num = pd.to_numeric(raw['Excess Readmission Ratio'], errors='coerce')
print("null ERR rows:", err_num.isna().sum())
print("footnote value counts:")
print(raw['Footnote'].value_counts(dropna=False))
print("\nFootnotes 1/5/7 total:", raw['Footnote'].isin(['1','5','7']).sum())
print("=> equals the null-ERR count, so suppression is fully explained (not random).")
print("Footnote 29 rows that still have ERR:",
      raw.loc[raw['Footnote']=='29','Excess Readmission Ratio'].apply(lambda x: pd.notna(pd.to_numeric(x, errors='coerce'))).sum(),
      "of", (raw['Footnote']=='29').sum(), "-> advisory, keep them")

null ERR rows: 6610
footnote value counts:
Footnote
NaN    11343
5       3255
1       3150
29       377
7        205
Name: count, dtype: int64

Footnotes 1/5/7 total: 6610
=> equals the null-ERR count, so suppression is fully explained (not random).
Footnote 29 rows that still have ERR: 377 of 377 -> advisory, keep them


### Cleaning decisions (see `docs/03_data_quality_log.md`)
1. Coerce numerics; map sentinels (`Not Available`, `Too Few to Report`, `N/A`, ``) → null.
2. Rows with footnote **1/5/7** = suppressed ERR → **not reported** (excluded from rate stats).
3. Rows with footnote **29** keep their valid ERR but are flagged `has_advisory`.
4. Keep `Facility ID` as string (preserve leading zeros).
5. Volume analysis limited to rows with both ERR and discharge count.

In [5]:
# The canonical cleaning lives in src/build_aggregates.py; we load its output here.
clean = pd.read_csv(PROC, dtype={'facility_id':str})
print("reported (usable ERR) rows:", len(clean))
print("with discharge volume:", clean['discharges'].notna().sum())
clean[['facility_id','facility_name','state','condition','err','discharges']].head()

reported (usable ERR) rows: 11720
with discharge volume: 8037


,facility_id,facility_name,state,condition,err,discharges
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,Hip/Knee Replacement,0.9875,NaN
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,Bypass Surgery,0.9531,137.0
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,Heart Attack,0.9370,273.0
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,COPD,0.9823,122.0
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,AL,Pneumonia,0.9871,507.0


✅ Clean, reported-only dataset ready for EDA → `notebooks/02_eda.ipynb`.